In [30]:
import pandas as pd
import numpy as np
from pycaret.classification import load_model, predict_model
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

blend = load_model("permdi_blend_ridge_cat_v1")  # ✓

# 1) Inspect the pipeline structure
print("Pipeline steps:", [name for name, _ in blend.steps])

# 2) Grab the trained VotingClassifier from the pipeline
#    (PyCaret names the last step 'trained_model' in v3.x)
try:
    voter = blend.named_steps["trained_model"]
except KeyError:
    # Fallbacks if the name differs in your env
    voter = getattr(blend, "_final_estimator", None) or blend.steps[-1][1]

print("Final estimator type:", type(voter))

# 3) List base estimators inside the VotingClassifier
#    (scikit-learn exposes .estimators_ after fit, or .estimators before fit)
base_ests = getattr(voter, "estimators_", None) or getattr(voter, "estimators", None)
print("Num base estimators:", len(base_ests))

# 4) Sanity check types (should be Calibrated Ridge and CatBoost)
print([type(e) for e in base_ests])

# Optional asserts to prove what’s inside:
assert any(isinstance(e, CalibratedClassifierCV) for e in base_ests), "Calibrated Ridge missing?"
assert any(isinstance(e, CatBoostClassifier) for e in base_ests), "CatBoost missing?"


Transformation Pipeline and Model Successfully Loaded
Pipeline steps: ['numerical_imputer', 'categorical_imputer', 'ordinal_encoding', 'onehot_encoding', 'balance', 'clean_column_names', 'actual_estimator']
Final estimator type: <class 'sklearn.ensemble._voting.VotingClassifier'>
Num base estimators: 2
[<class 'sklearn.calibration.CalibratedClassifierCV'>, <class 'catboost.core.CatBoostClassifier'>]


In [31]:
import pandas as pd
import numpy as np
from pycaret.classification import load_model, predict_model
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

recurrence = load_model("recurrence_nb_calibrated_v1")  # ✓

# 1) Inspect the pipeline structure
print("Pipeline steps:", [name for name, _ in blend.steps])

# 2) Grab the trained VotingClassifier from the pipeline
#    (PyCaret names the last step 'trained_model' in v3.x)
try:
    voter = blend.named_steps["trained_model"]
except KeyError:
    # Fallbacks if the name differs in your env
    voter = getattr(blend, "_final_estimator", None) or blend.steps[-1][1]

print("Final estimator type:", type(voter))

# 3) List base estimators inside the VotingClassifier
#    (scikit-learn exposes .estimators_ after fit, or .estimators before fit)
base_ests = getattr(voter, "estimators_", None) or getattr(voter, "estimators", None)
print("Num base estimators:", len(base_ests))

# 4) Sanity check types (should be Calibrated Ridge and CatBoost)
print([type(e) for e in base_ests])

# Optional asserts to prove what’s inside:
assert any(isinstance(e, CalibratedClassifierCV) for e in base_ests), "Calibrated Ridge missing?"
assert any(isinstance(e, CatBoostClassifier) for e in base_ests), "CatBoost missing?"


Transformation Pipeline and Model Successfully Loaded
Pipeline steps: ['numerical_imputer', 'categorical_imputer', 'ordinal_encoding', 'onehot_encoding', 'balance', 'clean_column_names', 'actual_estimator']
Final estimator type: <class 'sklearn.ensemble._voting.VotingClassifier'>
Num base estimators: 2
[<class 'sklearn.calibration.CalibratedClassifierCV'>, <class 'catboost.core.CatBoostClassifier'>]


In [32]:
df_raw.columns

Index(['Date', 'Age ', 'BMI', 'Surgical Approach',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Axes disrupted', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Tumor volume', 'Tumor (Adamantinomatous, papillary)', 'BRAF+',
       'Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)',
       'Intraventricular extension', 'Solid, cystic, mixed',
       'Resection outcome (total, subtotal, partial)', 'Prior operation',
       'recurring tumor?', 'Recurrence after this operation',
       'Recurrence date', 'Days to recurrence',
       'Re-operation after this operation (add date)', 'Time to r

In [33]:
# Minimal smoke test on a few rows from your original CSV
FEATURES = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Surgical Approach", "Resection outcome (total, subtotal, partial)"
]

def prepare_inputs(df):
    df = df.copy()
    # recompute tumor volume if needed
    if "Tumor volume" not in df.columns and all(c in df.columns for c in ["Tumor size","Tumor size.1","Tumor size.2"]):
        df["Tumor volume"] = (4/3)*np.pi * (df["Tumor size"]/2)*(df["Tumor size.1"]/2)*(df["Tumor size.2"]/2)
    # ensure all expected columns exist
    for c in FEATURES:
        if c not in df.columns:
            df[c] = np.nan
    # light coercion for binary-like cols
    bin_like = [
        "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
        "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
        "Pre-op headache", "Pre-op endocrine changes (y/n)",
        "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
        "Intraventricular extension",
    ]
    for c in bin_like:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # keep exact column order
    return df[FEATURES]

df_raw = pd.read_csv("data.csv")
X_new = prepare_inputs(df_raw).head(5)
pred1 = predict_model(blend, data=X_new)
pred2 = predict_model(recurrence, data=X_new)
print(pred1[["prediction_label","prediction_score"]])
print(pred2[["prediction_label","prediction_score"]])


   prediction_label  prediction_score
0                 0            0.6172
1                 0            0.7061
2                 1            0.5628
3                 1            0.6767
4                 1            0.5477
   prediction_label  prediction_score
0                 0            0.6506
1                 0            1.0000
2                 1            0.8820
3                 0            0.6333
4                 0            0.6333


In [36]:
import requests, textwrap, json

url = "https://precision-craniopharyngiomas.onrender.com/predict"
payload = {
    "Age ": 60, "BMI": 28.3,
    "Tumor size": 3.4, "Tumor size.1": 2.2, "Tumor size.2": 1.9,
    "Last BMI on file": 29.5,
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss": 1,
    "Pre-op hydrocephalus (y/n)": 0, "Pre-op intacranial pressure symptoms": 0,
    "Pre-op headache": 1, "Pre-op endocrine changes (y/n)": 1,
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional": 0,
    "Intraventricular extension": 0,
    "Surgical Approach": "craniotomy",
    "Resection outcome (total, subtotal, partial)": "subtotal"
}
r = requests.post(url, json=payload, timeout=20)
print("Status:", r.status_code)
print("Text:", r.text)           # read the human error if 400
try:
    print("JSON:", r.json())
except Exception:
    pass


Status: 200
Text: {"prob_permanent_DI":0.6258092026670323,"label":1,"threshold":0.5}
JSON: {'prob_permanent_DI': 0.6258092026670323, 'label': 1, 'threshold': 0.5}
